# Motex V4（已验证改进的默认组合 · notebook 内装配）

V4 = V3（内置因果掩码 / 权重绑定） + 通过 A/B 实验验证的改进，全部机制均有注释说明：

1. **MLA（Multi-head Latent Attention）**：KV 只缓存低维 latent（默认 32 维），KV 显存由
   GQA 的 2×kv×head_dim 降到 latent_dim → **约 1/8**；域内略优、外推更优（A/B 已验证）。
2. **QK-Norm（默认开）**：softmax 前对 Q/K 做 RMSNorm；域内最优 + 外推 CE 11.6→6.1，零成本。
3. **bf16/AMP 训练（推荐模式）**：吞吐 ≈2×、0 NaN（训练侧脚本见 dev/）。

可选（实验结论为负/场景性，默认关）：`rope_scaling`（RoPE 外推）、`use_sdp`（真 FlashAttention，
大模型/长上下文再开）。

> 与 v1/v2 一致：**模型装配写在 notebook 里**，`motex_utils` 只放共享构建块。

In [ ]:
# 仓库根目录加入 sys.path（本仓库自包含）
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
from torch import nn

# 共享构建块
from motex_utils.transformer import RMSNorm
from motex_utils.attention import MLAAttention   # 低秩 latent KV 注意力（改进①说明见其 docstring）
from motex_utils.moe import SwiGLUMLP

In [ ]:
# ============ 模型装配（notebook 内定义）============
class MotexV4Block(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads, ffn_hidden, dropout, i, max_seq_len,
                 mla_latent_dim=32, qk_norm=True, use_sdp=False):
        super().__init__()
        self.i = i
        self.norm1 = RMSNorm(d_model)
        self.attn = MLAAttention(d_model, num_heads, num_kv_heads, dropout, max_seq_len,
                                latent_dim=mla_latent_dim, use_sdp=use_sdp, qk_norm=qk_norm)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLUMLP(d_model, ffn_hidden, dropout)

    def forward(self, x, state=None, valid_lens=None):
        h = self.norm1(x)
        attn_out, state = self.attn(h, state, self.i)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, state, 0.0


class MotexV4Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, num_kv_heads,
                 ffn_hidden, dropout, max_seq_len, mla_latent_dim=32, qk_norm=True, use_sdp=False):
        super().__init__()
        self.num_layers = num_layers
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blks = nn.ModuleList([
            MotexV4Block(d_model, num_heads, num_kv_heads, ffn_hidden, dropout, i, max_seq_len,
                         mla_latent_dim=mla_latent_dim, qk_norm=qk_norm, use_sdp=use_sdp)
            for i in range(num_layers)
        ])

    def forward(self, tokens, valid_lens=None, state=None):
        x = self.token_emb(tokens)
        if state is not None and state[0] is None:
            state[0] = [None] * self.num_layers
        aux = 0.0
        for blk in self.blks:
            x, state, a = blk(x, state, valid_lens)
            aux += a
        return x, state, aux / len(self.blks)


class MotexV4(nn.Module):
    """统一接口：forward(tokens, valid_lens=None, state=None) -> (logits, state, aux_loss)"""
    def __init__(self, vocab_size, d_model=512, num_layers=8, num_heads=8, num_kv_heads=2,
                 ffn_hidden=1024, dropout=0.1, max_seq_len=256,
                 mla_latent_dim=32, qk_norm=True, use_sdp=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.decoder = MotexV4Decoder(vocab_size, d_model, num_layers, num_heads, num_kv_heads,
                                     ffn_hidden, dropout, max_seq_len,
                                     mla_latent_dim=mla_latent_dim, qk_norm=qk_norm, use_sdp=use_sdp)
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.decoder.token_emb.weight   # 权重绑定
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, tokens, valid_lens=None, state=None):
        x, state, aux = self.decoder(tokens, valid_lens, state)
        return self.lm_head(self.norm(x)), state, aux

In [ ]:
# ============ 生成（KV-Cache 增量解码，同 v3）============
from torch.nn import functional as F

@torch.no_grad()
def generate(net, tokenizer, prompt, max_new_tokens, device, temperature=0.8, top_k=40,
             repetition_penalty=1.15):
    net.eval()
    ids = tokenizer.encode(prompt)
    special = {tokenizer.stoi[s] for s in ('<pad>', '<bos>', '<eos>') if s in tokenizer.stoi}
    state = [[None] * net.decoder.num_layers, [None] * net.decoder.num_layers]
    logits, state, _ = net(torch.tensor([ids], device=device), None, state)

    def pick(logts, seen):
        logts = logts.clone()
        if repetition_penalty != 1.0 and seen:
            for t in seen:
                logts[t] /= repetition_penalty
        p = F.softmax(logts / temperature, dim=-1)
        top = torch.topk(p, min(top_k, p.numel()))
        return top.indices[torch.multinomial(top.values, 1)].item()

    seen = []; gen = []
    nxt = pick(logits[0, -1, :], seen)
    for _ in range(max_new_tokens):
        gen.append(nxt); seen.append(nxt)
        logits, state, _ = net(torch.tensor([[nxt]], device=device), None, state)
        nxt = pick(logits[0, -1, :], seen)
    return prompt + ''.join(tokenizer.itos[i] for i in gen if i not in special), gen

In [ ]:
# ============ 使用演示 ============
net = MotexV4(vocab_size=4096, d_model=256, num_layers=4, num_heads=4,
              num_kv_heads=2, ffn_hidden=512, dropout=0.1, max_seq_len=128)
print('参数(M):', round(sum(p.numel() for p in net.parameters()) / 1e6, 2))
print('注意力:', type(net.decoder.blks[0].attn).__name__, '| qk_norm =', net.decoder.blks[0].attn.qk_norm)
print('KV 缓存/层/token(B):', net.decoder.blks[0].attn.kv_cache_bytes_per_token())

x = torch.randint(5, 4096, (2, 128))
logits, state, aux = net(x, None, None)
print('train 前向 logits', tuple(logits.shape), ' 返回 3 元组')

## bf16 训练（推荐）
```python
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    logits, _, _ = net(x, None, None)
    l = loss(logits.reshape(-1, vocab), y.reshape(-1))
l.float().backward()   # bf16 不需要 GradScaler
```